# CNM Analysis: LHC p+Pb (5.02 & 8.16 TeV)

This notebook calculates Coherent Nuclear Matter (CNM) effects for charm production in p+Pb collisions at the LHC.
Components include:
* **nPDF**: Nuclear Parton Distribution Function effects (EPPS21)
* **Eloss**: Coherent energy loss
* **Broad**: $p_T$ broadening (Cronin effect)
* **CNM**: $nPDF \times (Eloss \times Broad)$

## 1. Setup and Imports

In [1]:
import sys
import time
from pathlib import Path
import numpy as np
import os

# Add repo paths
HERE = Path(".").resolve()
sys.path.append(str(HERE.parent / "eloss_code"))
sys.path.append(str(HERE.parent / "npdf_code"))
sys.path.append(str(HERE.parent / "cnm_combine"))

from cnm_combine_fast import CNMCombineFast
from cnm_combine import make_centrality_weight_dict
import matplotlib.pyplot as plt

print("Libraries loaded.")


Libraries loaded.


## 2. Helper Plotting Functions

In [2]:
def plot_bands(x, bands, labels, xlabel, ylabel="$R_{pA}$", title="", filename=None):
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(labels)))
    color_map = {lab: col for lab, col in zip(labels, colors)}
    color_map["MB"] = "black"

    # 1. Main Plot: CNM Total (All Centralities)
    for lab in labels + ["MB"]:
        Dc, Dlo, Dhi = bands["cnm"]
        if lab in Dc:
            c = color_map.get(lab, "k")
            ax.plot(x, Dc[lab], color=c, label=lab, linewidth=2)
            ax.fill_between(x, Dlo[lab], Dhi[lab], color=c, alpha=0.2)
        
    ax.set_ylim(0, 1.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title + " (Total CNM)")
    ax.legend(title="Centrality", ncol=2)
    ax.axhline(1, color='k', ls=':', lw=1)
    
    if filename:
        fig.savefig(str(filename).replace(".png", "_Total.png"), dpi=150)
        
    # 2. Decomposition Plot (MinBias Only)
    fig2, ax2 = plt.subplots(figsize=(10, 7))
    comp_defs = [
        ("eloss", "Eloss", "k", "-", 0.5), 
        ("broad", "Broad", "k", "--", 0.5), 
        ("eloss_broad", "Eloss x Broad", "gray", "-", 1.0),
        ("npdf", "nPDF", "magenta", "-", 1.0),
        ("cnm", "Total CNM", "blue", "-", 1.0)
    ]
    
    lab = "MB"
    for comp, name, col, ls, alph in comp_defs:
        if comp in bands:
            Dc, Dlo, Dhi = bands[comp]
            if lab in Dc:
                ax2.plot(x, Dc[lab], color=col, linestyle=ls, alpha=alph, label=name, linewidth=2)
                ax2.fill_between(x, Dlo[lab], Dhi[lab], color=col, alpha=0.1)

    ax2.set_ylim(0, 1.5)
    ax2.set_xlabel(xlabel)
    ax2.set_ylabel(ylabel)
    ax2.set_title(title + " (MinBias Decomposition)")
    ax2.legend()
    ax2.axhline(1, color='k', ls=':', lw=1)
    
    if filename:
        fig2.savefig(str(filename).replace(".png", "_MB_Decomp.png"), dpi=150)
    return fig, fig2

def plot_cent_bands(cnm_cent_data, labels, title="", filename=None, ylabel="$R_{pA}$"):
    fig, ax = plt.subplots(figsize=(10, 7))
    x = np.arange(len(labels))
    comp_defs = [
        ("eloss", "Eloss", "k", "-", 0.5), 
        ("broad", "Broad", "k", "--", 0.5), 
        ("eloss_broad", "Eloss x Broad", "gray", "-", 1.0), 
        ("npdf", "nPDF", "magenta", "-", 1.0),
        ("cnm", "Total CNM", "blue", "-", 1.0)
    ]
    for comp, name, col, ls, alph in comp_defs:
        if comp not in cnm_cent_data: continue
        (Vc, Vlo, Vhi, MBc, MBlo, MBhi) = cnm_cent_data[comp]
        ax.axhline(MBc, color=col, linestyle="--", linewidth=1.5, alpha=alph)
        ax.axhspan(MBlo, MBhi, color=col, alpha=0.1)
        shift = 0.05 * (comp_defs.index((comp, name, col, ls, alph)) - 2)
        ax.errorbar(x + shift, Vc, yerr=[Vc-Vlo, Vhi-Vc], fmt='o-', color=col, label=name, alpha=alph, capsize=4)
        
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel("Centrality Bin")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_ylim(0.0, 1.4)
    ax.legend(ncol=2)
    ax.axhline(1, color='k', ls=':', lw=1)
    if filename: fig.savefig(filename, dpi=150)
    return fig


## 3. Analysis for 5.02 TeV

In [3]:
energy = "5.02"
system = "LHC"
print(f"\n{'='*60}")
print(f"STARTING ANALYSIS FOR {system} @ {energy} TeV" if energy != "200" else f"STARTING ANALYSIS FOR {system} @ {energy} GeV")
print(f"{'='*60}")

OUTDIR = Path(f"outputs/{system}_{energy}")
OUTDIR.mkdir(parents=True, exist_ok=True)

start_total = time.time()

# ----------------------------------------------------------------
# 3.1 Initialize
# ----------------------------------------------------------------
print(f"Step 1: Initializing CNM Pipeline...")
t0 = time.time()
y_windows_rhic = [(-2.2, -1.2, "d-going"), (-0.35, 0.35, "Mid-rap"), (1.2, 2.2, "Au-going")]
y_windows_lhc = [(-4.46, -2.96, "Backward"), (-1.37, 0.43, "Mid"), (2.03, 3.53, "Forward")]
y_win = y_windows_rhic if energy == "200" else y_windows_lhc

cnm = CNMCombineFast.from_defaults(
    energy=energy,
    family="charmonia",
    particle_state="avg",
    cent_bins=[(0,20), (20,40), (40,60), (60,100)],
    y_windows=y_win
)
print(f"  -> Done in {time.time()-t0:.2f}s")
labels = [f"{int(a)}-{int(b)}%" for (a,b) in cnm.cent_bins]

# ----------------------------------------------------------------
# 3.2 RpA vs Rapidity
# ----------------------------------------------------------------
print(f"Step 2: Calculating components vs Rapidity...")
t_y = time.time()

# 3.2.1 nPDF vs y
print("  -> calculating nPDF vs y...")
t_n = time.time()
from npdf_centrality import bin_rpa_vs_y, bin_rpa_vs_pT, bin_rpa_vs_centrality, make_centrality_weight_dict
if energy == "200": from npdf_centrality_dAu import bin_rpa_vs_y, bin_rpa_vs_pT, bin_rpa_vs_centrality

wcent = make_centrality_weight_dict(cnm.cent_bins, c0=cnm.cent_c0)

npdf_data_y = bin_rpa_vs_y(
    cnm.npdf_ctx['df49_by_cent'], 
    cnm.npdf_ctx['df_pp'], 
    cnm.npdf_ctx['df_pa'], 
    cnm.npdf_ctx['gluon'],
    cent_bins=cnm.cent_bins,
    y_edges=cnm.y_edges,
    pt_range_avg=cnm.pt_range_avg,
    wcent_dict=wcent
)
print(f"     (nPDF y done in {time.time()-t_n:.2f}s)")

# 3.2.2 Eloss+Broad vs y
print("  -> calculating Eloss+Broad vs y...")
t_e = time.time()
y_cent, e_bands_y, _ = cnm._calc_eloss_broad_band_vs_y(cnm.y_edges, cnm.pt_range_avg, ["loss","broad","eloss_broad"])
print(f"     (Eloss y done in {time.time()-t_e:.2f}s)")

# 3.2.3 Combine
bands_y = cnm._combine_bands_generic(["npdf","eloss","broad","eloss_broad","cnm"], npdf_data_y, e_bands_y, labels, True, mode="y")

plot_bands(y_cent, bands_y, labels, "$y_{CMS}$", title=f"{system} {energy}: RpA vs y", filename=OUTDIR / "RpA_vs_y.png")
print(f"  -> Step 2 Total Done in {time.time()-t_y:.2f}s")

# ----------------------------------------------------------------
# 3.3 RpA vs pT
# ----------------------------------------------------------------
print(f"Step 3: Calculating components vs pT...")
t_pt_all = time.time()
for (y0, y1, desc) in cnm.y_windows:
    print(f"  -> Rapidity window: {desc} ({y0} to {y1})")
    t_win = time.time()
    
    # nPDF
    npdf_pt = bin_rpa_vs_pT(
        cnm.npdf_ctx['df49_by_cent'], 
        cnm.npdf_ctx['df_pp'], 
        cnm.npdf_ctx['df_pa'], 
        cnm.npdf_ctx['gluon'],
        cent_bins=cnm.cent_bins,
        pt_edges=cnm.p_edges,
        y_window=(y0, y1),
        wcent_dict=wcent
    )
    
    # Eloss
    pt_cent, e_bands_pt, _ = cnm._calc_eloss_broad_band_vs_pT(cnm.p_edges, (y0, y1), ["loss","broad","eloss_broad"])
    
    # Combine
    bands_pt = cnm._combine_bands_generic(["npdf","eloss","broad","eloss_broad","cnm"], npdf_pt, e_bands_pt, labels, True, mode="pT")
    
    plot_bands(pt_cent, bands_pt, labels, "$p_T$ [GeV]", title=f"{system} {energy}: {desc}", filename=OUTDIR / f"RpA_vs_pT_{y0}_{y1}.png")
    plt.close('all')
    print(f"     (window done in {time.time()-t_win:.2f}s)")

print(f"  -> Step 3 Total Done in {time.time()-t_pt_all:.2f}s")

# ----------------------------------------------------------------
# 3.4 RpA vs Centrality
# ----------------------------------------------------------------
print(f"Step 4: Calculating components vs Centrality...")
t_cent_all = time.time()
for (y0, y1, desc) in cnm.y_windows:
    print(f"  -> Rapidity window: {desc} ({y0} to {y1})")
    t_win = time.time()
    
    res = cnm.cnm_vs_centrality(y_window=(y0, y1))
    plot_cent_bands(res, labels, title=f"{system} {energy} Centrality: {desc}", filename=OUTDIR / f"RpA_vs_cent_{y0}_{y1}.png", ylabel=f"$R_{{{ "dAu" if energy=="200" else "pPb" }}}$")
    plt.close('all')
    print(f"     (window done in {time.time()-t_win:.2f}s)")

print(f"  -> Step 4 Total Done in {time.time()-t_cent_all:.2f}s")
print(f"\nTotal analysis time for {energy}: {time.time()-start_total:.2f}s")



STARTING ANALYSIS FOR LHC @ 5.02 TeV
Step 1: Initializing CNM Pipeline...
  -> Done in 9.57s
Step 2: Calculating components vs Rapidity...
  -> calculating nPDF vs y...
     (nPDF y done in 0.27s)
  -> calculating Eloss+Broad vs y...
     (Eloss y done in 10.87s)
  -> Step 2 Total Done in 11.42s
Step 3: Calculating components vs pT...
  -> Rapidity window: Backward (-4.46 to -2.96)
     (window done in 5.90s)
  -> Rapidity window: Mid (-1.37 to 0.43)
     (window done in 6.03s)
  -> Rapidity window: Forward (2.03 to 3.53)
     (window done in 6.34s)
  -> Step 3 Total Done in 18.27s
Step 4: Calculating components vs Centrality...
  -> Rapidity window: Backward (-4.46 to -2.96)
     (window done in 0.65s)
  -> Rapidity window: Mid (-1.37 to 0.43)
     (window done in 0.66s)
  -> Rapidity window: Forward (2.03 to 3.53)
     (window done in 0.66s)
  -> Step 4 Total Done in 1.97s

Total analysis time for 5.02: 41.23s


## 4. Analysis for 8.16 TeV

In [4]:
energy = "8.16"
system = "LHC"
print(f"\n{'='*60}")
print(f"STARTING ANALYSIS FOR {system} @ {energy} TeV" if energy != "200" else f"STARTING ANALYSIS FOR {system} @ {energy} GeV")
print(f"{'='*60}")

OUTDIR = Path(f"outputs/{system}_{energy}")
OUTDIR.mkdir(parents=True, exist_ok=True)

start_total = time.time()

# ----------------------------------------------------------------
# 3.1 Initialize
# ----------------------------------------------------------------
print(f"Step 1: Initializing CNM Pipeline...")
t0 = time.time()
y_windows_rhic = [(-2.2, -1.2, "d-going"), (-0.35, 0.35, "Mid-rap"), (1.2, 2.2, "Au-going")]
y_windows_lhc = [(-4.46, -2.96, "Backward"), (-1.37, 0.43, "Mid"), (2.03, 3.53, "Forward")]
y_win = y_windows_rhic if energy == "200" else y_windows_lhc

cnm = CNMCombineFast.from_defaults(
    energy=energy,
    family="charmonia",
    particle_state="avg",
    cent_bins=[(0,20), (20,40), (40,60), (60,100)],
    y_windows=y_win
)
print(f"  -> Done in {time.time()-t0:.2f}s")
labels = [f"{int(a)}-{int(b)}%" for (a,b) in cnm.cent_bins]

# ----------------------------------------------------------------
# 3.2 RpA vs Rapidity
# ----------------------------------------------------------------
print(f"Step 2: Calculating components vs Rapidity...")
t_y = time.time()

# 3.2.1 nPDF vs y
print("  -> calculating nPDF vs y...")
t_n = time.time()
from npdf_centrality import bin_rpa_vs_y, bin_rpa_vs_pT, bin_rpa_vs_centrality, make_centrality_weight_dict
if energy == "200": from npdf_centrality_dAu import bin_rpa_vs_y, bin_rpa_vs_pT, bin_rpa_vs_centrality

wcent = make_centrality_weight_dict(cnm.cent_bins, c0=cnm.cent_c0)

npdf_data_y = bin_rpa_vs_y(
    cnm.npdf_ctx['df49_by_cent'], 
    cnm.npdf_ctx['df_pp'], 
    cnm.npdf_ctx['df_pa'], 
    cnm.npdf_ctx['gluon'],
    cent_bins=cnm.cent_bins,
    y_edges=cnm.y_edges,
    pt_range_avg=cnm.pt_range_avg,
    wcent_dict=wcent
)
print(f"     (nPDF y done in {time.time()-t_n:.2f}s)")

# 3.2.2 Eloss+Broad vs y
print("  -> calculating Eloss+Broad vs y...")
t_e = time.time()
y_cent, e_bands_y, _ = cnm._calc_eloss_broad_band_vs_y(cnm.y_edges, cnm.pt_range_avg, ["loss","broad","eloss_broad"])
print(f"     (Eloss y done in {time.time()-t_e:.2f}s)")

# 3.2.3 Combine
bands_y = cnm._combine_bands_generic(["npdf","eloss","broad","eloss_broad","cnm"], npdf_data_y, e_bands_y, labels, True, mode="y")

plot_bands(y_cent, bands_y, labels, "$y_{CMS}$", title=f"{system} {energy}: RpA vs y", filename=OUTDIR / "RpA_vs_y.png")
print(f"  -> Step 2 Total Done in {time.time()-t_y:.2f}s")

# ----------------------------------------------------------------
# 3.3 RpA vs pT
# ----------------------------------------------------------------
print(f"Step 3: Calculating components vs pT...")
t_pt_all = time.time()
for (y0, y1, desc) in cnm.y_windows:
    print(f"  -> Rapidity window: {desc} ({y0} to {y1})")
    t_win = time.time()
    
    # nPDF
    npdf_pt = bin_rpa_vs_pT(
        cnm.npdf_ctx['df49_by_cent'], 
        cnm.npdf_ctx['df_pp'], 
        cnm.npdf_ctx['df_pa'], 
        cnm.npdf_ctx['gluon'],
        cent_bins=cnm.cent_bins,
        pt_edges=cnm.p_edges,
        y_window=(y0, y1),
        wcent_dict=wcent
    )
    
    # Eloss
    pt_cent, e_bands_pt, _ = cnm._calc_eloss_broad_band_vs_pT(cnm.p_edges, (y0, y1), ["loss","broad","eloss_broad"])
    
    # Combine
    bands_pt = cnm._combine_bands_generic(["npdf","eloss","broad","eloss_broad","cnm"], npdf_pt, e_bands_pt, labels, True, mode="pT")
    
    plot_bands(pt_cent, bands_pt, labels, "$p_T$ [GeV]", title=f"{system} {energy}: {desc}", filename=OUTDIR / f"RpA_vs_pT_{y0}_{y1}.png")
    plt.close('all')
    print(f"     (window done in {time.time()-t_win:.2f}s)")

print(f"  -> Step 3 Total Done in {time.time()-t_pt_all:.2f}s")

# ----------------------------------------------------------------
# 3.4 RpA vs Centrality
# ----------------------------------------------------------------
print(f"Step 4: Calculating components vs Centrality...")
t_cent_all = time.time()
for (y0, y1, desc) in cnm.y_windows:
    print(f"  -> Rapidity window: {desc} ({y0} to {y1})")
    t_win = time.time()
    
    res = cnm.cnm_vs_centrality(y_window=(y0, y1))
    plot_cent_bands(res, labels, title=f"{system} {energy} Centrality: {desc}", filename=OUTDIR / f"RpA_vs_cent_{y0}_{y1}.png", ylabel=f"$R_{{{ "dAu" if energy=="200" else "pPb" }}}$")
    plt.close('all')
    print(f"     (window done in {time.time()-t_win:.2f}s)")

print(f"  -> Step 4 Total Done in {time.time()-t_cent_all:.2f}s")
print(f"\nTotal analysis time for {energy}: {time.time()-start_total:.2f}s")



STARTING ANALYSIS FOR LHC @ 8.16 TeV
Step 1: Initializing CNM Pipeline...
  -> Done in 10.41s
Step 2: Calculating components vs Rapidity...
  -> calculating nPDF vs y...
     (nPDF y done in 0.29s)
  -> calculating Eloss+Broad vs y...
     (Eloss y done in 11.14s)
  -> Step 2 Total Done in 11.58s
Step 3: Calculating components vs pT...
  -> Rapidity window: Backward (-4.46 to -2.96)
     (window done in 6.35s)
  -> Rapidity window: Mid (-1.37 to 0.43)
     (window done in 5.95s)
  -> Rapidity window: Forward (2.03 to 3.53)
     (window done in 5.89s)
  -> Step 3 Total Done in 18.19s
Step 4: Calculating components vs Centrality...
  -> Rapidity window: Backward (-4.46 to -2.96)
     (window done in 0.62s)
  -> Rapidity window: Mid (-1.37 to 0.43)
     (window done in 0.64s)
  -> Rapidity window: Forward (2.03 to 3.53)
     (window done in 0.62s)
  -> Step 4 Total Done in 1.88s

Total analysis time for 8.16: 42.06s
